In [1]:
#导入库
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='tqdm')

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict
import random
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 库导入成功")


/root/miniconda3/envs/log-anomaly/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 库导入成功


In [4]:
# 加载结构化日志数据
structured_df = pd.read_csv('../data/loghub/BGL/BGL_2k.log_structured.csv')

# 添加二值标签
structured_df['is_anomaly'] = structured_df['Label'] != '-'

print(f"数据加载完成")
print(f"总样本数: {len(structured_df)}")
print(f"列名: {structured_df.columns.tolist()}")
print(f"\n前3行:")
structured_df.head(3)


数据加载完成
总样本数: 2000
列名: ['LineId', 'Label', 'Timestamp', 'Date', 'Node', 'Time', 'NodeRepeat', 'Type', 'Component', 'Level', 'Content', 'EventId', 'EventTemplate', 'is_anomaly']

前3行:


,LineId,Label,Timestamp,Date,Node,Time,NodeRepeat,Type,Component,Level,Content,EventId,EventTemplate,is_anomaly
0,1,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E77,instruction cache parity error corrected,False
1,2,-,1117838573,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.53.276129,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E77,instruction cache parity error corrected,False
2,3,-,1117838976,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.49.36.156884,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,E77,instruction cache parity error corrected,False


In [9]:
#查看标签分布
print("标签分布:")
print(structured_df['is_anomaly'].value_counts())
print(f"\n异常率: {structured_df['is_anomaly'].mean():.2%}")

标签分布:
is_anomaly
False    1857
True      143
Name: count, dtype: int64

异常率: 7.15%


构建数据集和测试集代码

In [14]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# 定义输出目录和文件路径
output_dir = "/root/autodl-tmp/log-anomaly-detection/output"
train_file_path = os.path.join(output_dir, "historical_df.csv")  # 训练集文件
test_file_path = os.path.join(output_dir, "test_df.csv")        # 测试集文件

# 核心作用：按 7:3 比例拆分数据集，同时保证拆分后训练集 / 测试集的异常日志比例和原数据一致
historical_df, test_df = train_test_split(
    structured_df,
    test_size=0.3,
    random_state=42,
    stratify=structured_df['is_anomaly']
)

# ========== 新增：保存数据集到指定目录 ==========
# 创建输出目录（如果不存在）
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"📁 创建输出目录: {output_dir}")

# 保存训练集
historical_df.to_csv(train_file_path, index=False, encoding='utf-8')
print(f"💾 训练集已保存至: {train_file_path}")

# 保存测试集
test_df.to_csv(test_file_path, index=False, encoding='utf-8')
print(f"💾 测试集已保存至: {test_file_path}")

# 打印数据集统计信息
print(f"\n历史数据集: {len(historical_df)} 条")
print(f"  - 正常: {(~historical_df['is_anomaly']).sum()}")
print(f"  - 异常: {historical_df['is_anomaly'].sum()}")

print(f"\n测试数据集: {len(test_df)} 条")
print(f"  - 正常: {(~test_df['is_anomaly']).sum()}")
print(f"  - 异常: {test_df['is_anomaly'].sum()}")


💾 训练集已保存至: /root/autodl-tmp/log-anomaly-detection/output/historical_df.csv
💾 测试集已保存至: /root/autodl-tmp/log-anomaly-detection/output/test_df.csv

历史数据集: 1400 条
  - 正常: 1300
  - 异常: 100

测试数据集: 600 条
  - 正常: 557
  - 异常: 43


在训练集不同模板中找正常、异常样本（各两个）

# 1.生成Kh的代码
Kh筛选原则：对每一个模板的日志中，正常的选20条,异常的选20条（如果没有20条选剩下的全部即可）

In [1]:
# 修正版本 - CSV包含模板、标签和原始内容
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import random
import os
import csv

def quick_template_based_sampling(log_file, output_file, 
                                   samples_per_template_normal=20, 
                                   samples_per_template_anomaly=20):
    """基于模板的采样 - 输出包含模板和原始内容的CSV"""
    
    print("🔍 基于模板的采样...")
    print(f"   策略: 每个模板采样 {samples_per_template_normal} 条正常 + {samples_per_template_anomaly} 条异常")
    
    # 初始化 Drain
    config = TemplateMinerConfig()
    config.drain_extra_delimiters = []
    config.drain_sim_th = 0.4
    config.drain_depth = 4
    config.drain_max_children = 100
    config.drain_max_clusters = 1024
    
    miner = TemplateMiner(config=config)
    
    # 解析日志 - 按模板和标签分组
    template_groups = {}  # {template_id: {'normal': [], 'anomaly': []}}
    cluster_map = {}
    
    print(f"📂 读取文件: {log_file}")
    
    with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line_num, line in enumerate(f, 1):
            if line_num % 100000 == 0:
                print(f"⏳ 已处理 {line_num:,} 行...")
            
            line = line.strip()
            if not line:
                continue
            
            parts = line.split(None, 9)
            if len(parts) < 10:
                continue
            
            # 判断是否异常
            label = parts[0]
            is_anomaly = not label.startswith('-')
            
            content = parts[9]
            
            # 使用 Drain 提取模板
            result = miner.add_log_message(content)
            template_id = result["cluster_id"]
            
            # 保存 cluster 对象
            if template_id not in cluster_map:
                for cluster in miner.drain.clusters:
                    if cluster.cluster_id == template_id:
                        cluster_map[template_id] = cluster
                        break
            
            log_entry = {
                'line': line,
                'template_id': template_id,
                'content': content,
                'is_anomaly': is_anomaly,
                'label': 'anomaly' if is_anomaly else 'normal'
            }
            
            # 按模板和标签分组
            if template_id not in template_groups:
                template_groups[template_id] = {'normal': [], 'anomaly': []}
            
            if is_anomaly:
                template_groups[template_id]['anomaly'].append(log_entry)
            else:
                template_groups[template_id]['normal'].append(log_entry)
    
    print(f"📊 解析完成:")
    print(f"   模板总数: {len(template_groups)}")
    
    # 统计每个模板的正常和异常数量
    total_normal = sum(len(g['normal']) for g in template_groups.values())
    total_anomaly = sum(len(g['anomaly']) for g in template_groups.values())
    print(f"   总正常日志: {total_normal:,}")
    print(f"   总异常日志: {total_anomaly:,}")
    
    # 统计模板分布
    template_stats = []
    templates_with_both = 0
    templates_only_normal = 0
    templates_only_anomaly = 0
    
    for tid, groups in template_groups.items():
        n_normal = len(groups['normal'])
        n_anomaly = len(groups['anomaly'])
        template_stats.append((tid, n_normal, n_anomaly))
        
        if n_normal > 0 and n_anomaly > 0:
            templates_with_both += 1
        elif n_normal > 0:
            templates_only_normal += 1
        else:
            templates_only_anomaly += 1
    
    print(f"📋 模板分类:")
    print(f"   同时有正常和异常: {templates_with_both}")
    print(f"   仅有正常: {templates_only_normal}")
    print(f"   仅有异常: {templates_only_anomaly}")
    
    # 显示 Top 10 模板
    template_stats.sort(key=lambda x: x[1] + x[2], reverse=True)
    print(f"📋 Top 10 最频繁模板:")
    for i, (tid, n_normal, n_anomaly) in enumerate(template_stats[:10], 1):
        cluster = cluster_map.get(tid)
        if cluster:
            template_str = ' '.join(cluster.log_template_tokens)
        else:
            template_str = "Unknown"
        total = n_normal + n_anomaly
        print(f"   {i}. [ID:{tid}] 总:{total:,} | 正常:{n_normal:,} | 异常:{n_anomaly:,}")
        print(f"      {template_str[:80]}...")
    
    # 每个模板采样固定数量
    print(f"🎯 开始采样...")
    samples_by_template = {}  # 按模板ID组织样本
    
    for template_id, groups in template_groups.items():
        n_normal = len(groups['normal'])
        n_anomaly = len(groups['anomaly'])
        
        template_samples = []
        
        # 采样正常日志
        if n_normal > 0:
            n_sample = min(samples_per_template_normal, n_normal)
            sampled_normal = random.sample(groups['normal'], n_sample)
            template_samples.extend(sampled_normal)
        
        # 采样异常日志
        if n_anomaly > 0:
            n_sample = min(samples_per_template_anomaly, n_anomaly)
            sampled_anomaly = random.sample(groups['anomaly'], n_sample)
            template_samples.extend(sampled_anomaly)
        
        if template_samples:
            samples_by_template[template_id] = template_samples
    
    total_samples = sum(len(samples) for samples in samples_by_template.values())
    print(f"   采样完成: {total_samples} 个样本")
    
    # 保存为CSV格式 - 包含模板信息
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    # 按template_id排序
    sorted_template_ids = sorted(samples_by_template.keys())
    
    print(f"💾 保存为CSV...")
    
    with open(output_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        # 写入表头：template_id, label, template, content
        writer.writerow(['template_id', 'label', 'template', 'content'])
        
        # 按template_id分组写入
        for template_id in sorted_template_ids:
            samples = samples_by_template[template_id]
            
            # 获取该模板的聚合模板字符串
            cluster = cluster_map.get(template_id)
            if cluster:
                template_str = ' '.join(cluster.log_template_tokens)
            else:
                template_str = "Unknown"
            
            # 在每个模板内，先写正常再写异常
            samples.sort(key=lambda x: (x['is_anomaly'], x['content']))
            
            for sample in samples:
                writer.writerow([
                    sample['template_id'],
                    sample['label'],
                    template_str,  # 聚合的模板
                    sample['content']  # 原始日志内容
                ])
    
    print(f"✅ 保存完成")
    print(f"💾 文件: {output_file}")
    
    # 详细统计
    all_samples = []
    for samples in samples_by_template.values():
        all_samples.extend(samples)
    
    print(f"\n📊 采样统计:")
    print(f"   总样本数: {len(all_samples)}")
    
    normal_count = sum(1 for s in all_samples if not s['is_anomaly'])
    anomaly_count = sum(1 for s in all_samples if s['is_anomaly'])
    print(f"   正常样本: {normal_count} ({normal_count/len(all_samples)*100:.2f}%)")
    print(f"   异常样本: {anomaly_count} ({anomaly_count/len(all_samples)*100:.2f}%)")
    
    template_coverage = len(samples_by_template)
    print(f"   覆盖模板数: {template_coverage}/{len(template_groups)} ({template_coverage/len(template_groups)*100:.2f}%)")
    
    # 统计每个模板的采样情况
    template_sample_stats = {}
    for tid, samples in samples_by_template.items():
        template_sample_stats[tid] = {'normal': 0, 'anomaly': 0}
        for sample in samples:
            if sample['is_anomaly']:
                template_sample_stats[tid]['anomaly'] += 1
            else:
                template_sample_stats[tid]['normal'] += 1
    
    # 显示采样分布
    print(f"📋 采样分布 (Top 10):")
    sorted_templates = sorted(
        template_sample_stats.items(),
        key=lambda x: x[1]['normal'] + x[1]['anomaly'],
        reverse=True
    )
    
    for i, (tid, stats) in enumerate(sorted_templates[:10], 1):
        total = stats['normal'] + stats['anomaly']
        cluster = cluster_map.get(tid)
        if cluster:
            template_str = ' '.join(cluster.log_template_tokens)[:60]
        else:
            template_str = "Unknown"
        print(f"   {i}. [ID:{tid}] 采样:{total} (正常:{stats['normal']}, 异常:{stats['anomaly']})")
        print(f"      模板: {template_str}...")
    
    # 显示CSV格式示例
    print(f"📄 CSV格式示例 (前10行):")
    print(f"   template_id | label | template | content")
    print(f"   " + "-" * 100)
    
    # 显示前10条记录
    count = 0
    for template_id in sorted_template_ids:
        if count >= 10:
            break
        
        samples = samples_by_template[template_id]
        cluster = cluster_map.get(template_id)
        if cluster:
            template_str = ' '.join(cluster.log_template_tokens)[:40]
        else:
            template_str = "Unknown"
        
        for sample in samples:
            if count >= 10:
                break
            
            content_preview = sample['content'][:40] + "..." if len(sample['content']) > 40 else sample['content']
            print(f"   {sample['template_id']:3d} | {sample['label']:7s} | {template_str:40s} | {content_preview}")
            count += 1
    
    # 显示模板对比示例
    print(f"📋 模板对比示例 (同一模板的不同实例):")
    
    # 选择一个有多个样本的模板
    for template_id in sorted_template_ids:
        samples = samples_by_template[template_id]
        if len(samples) >= 3:
            cluster = cluster_map.get(template_id)
            if cluster:
                template_str = ' '.join(cluster.log_template_tokens)
            else:
                template_str = "Unknown"
            
            print(f"\n   模板 {template_id}: {template_str}")
            print(f"   原始日志实例:")
            
            for i, sample in enumerate(samples[:3], 1):
                print(f"      {i}. [{sample['label']}] {sample['content'][:80]}...")
            
            break
    
    return samples_by_template


# ==================== 使用示例 ====================

if __name__ == "__main__":
    # 设置随机种子（可选，用于结果可复现）
    random.seed(42)
    
    # 文件路径
    log_file = "/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/BGL.log"
    output_file = "/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Kh_20k_template_based.csv"
    
    # 每个模板采样 20 条正常 + 20 条异常
    samples = quick_template_based_sampling(
        log_file, 
        output_file,
        samples_per_template_normal=20,
        samples_per_template_anomaly=20
    )
    
    print("\n" + "="*100)
    print("🎉 采样完成！")
    print("="*100)


🔍 基于模板的采样...
   策略: 每个模板采样 20 条正常 + 20 条异常
📂 读取文件: /root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/BGL.log
⏳ 已处理 100,000 行...
⏳ 已处理 200,000 行...
⏳ 已处理 300,000 行...
⏳ 已处理 400,000 行...
⏳ 已处理 500,000 行...
⏳ 已处理 600,000 行...
⏳ 已处理 700,000 行...
⏳ 已处理 800,000 行...
⏳ 已处理 900,000 行...
⏳ 已处理 1,000,000 行...
⏳ 已处理 1,100,000 行...
⏳ 已处理 1,200,000 行...
⏳ 已处理 1,300,000 行...
⏳ 已处理 1,400,000 行...
⏳ 已处理 1,500,000 行...
⏳ 已处理 1,600,000 行...
⏳ 已处理 1,700,000 行...
⏳ 已处理 1,800,000 行...
⏳ 已处理 1,900,000 行...
⏳ 已处理 2,000,000 行...
⏳ 已处理 2,100,000 行...
⏳ 已处理 2,200,000 行...
⏳ 已处理 2,300,000 行...
⏳ 已处理 2,400,000 行...
⏳ 已处理 2,500,000 行...
⏳ 已处理 2,600,000 行...
⏳ 已处理 2,700,000 行...
⏳ 已处理 2,800,000 行...
⏳ 已处理 2,900,000 行...
⏳ 已处理 3,000,000 行...
⏳ 已处理 3,100,000 行...
⏳ 已处理 3,200,000 行...
⏳ 已处理 3,300,000 行...
⏳ 已处理 3,400,000 行...
⏳ 已处理 3,500,000 行...
⏳ 已处理 3,600,000 行...
⏳ 已处理 3,700,000 行...
⏳ 已处理 3,800,000 行...
⏳ 已处理 3,900,000 行...
⏳ 已处理 4,000,000 行...
⏳ 已处理 4,100,000 行...
⏳ 已处理 4,200,000 行...
⏳ 已处理 4,300,000 行...
⏳

# 2.生成Ke的代码
Ke生成原则：
1.同一个模板下日志出现的次数>10(目的：1.罕见模板往往是偶然事件，对检测效果的贡献率低/2.降低计算与存储成本/3.避免引入噪声干扰)

2.每个模板（含模糊模板）最终仅选1~2条样例（1条正常+1条异常）

In [1]:
import csv
from collections import defaultdict
import csv
from collections import defaultdict

def generate_ke_from_kh(kh_file, ke_output_file, min_frequency=10):
    """
    从 Kh 文件中生成 Ke
    
    策略：
    1. 只保留高频模板（出现次数 > min_frequency）
    2. 每个模板最多选 1-2 条样例：
       - 混合模板：1条正常 + 1条异常
       - 纯异常模板：1条异常
       - 纯正常模板：1条正常
    
    参数：
        kh_file: Kh CSV文件路径
        ke_output_file: Ke 输出文件路径
        min_frequency: 最小频率阈值（默认10）
    """
    
    print("🔍 从 Kh 生成 Ke...")
    print(f"   输入文件: {kh_file}")
    print(f"   最小频率阈值: {min_frequency}")
    print(f"   采样策略: 每个模板最多 1-2 条样例")
    
    # 第一步：读取 Kh 文件，按模板分组
    template_groups = defaultdict(lambda: {'normal': [], 'anomaly': []})
    
    print(f"\n📂 读取 Kh 文件...")
    with open(kh_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            template_id = int(row['template_id'])
            label = row['label']
            content = row['content']
            
            template_groups[template_id][label].append({
                'template_id': template_id,
                'label': label,
                'content': content
            })
    
    print(f"   读取完成: {len(template_groups)} 个模板")
    
    # 第二步：分析每个模板的类型
    print(f"📊 分析模板类型...")
    
    template_analysis = {}
    for template_id, groups in template_groups.items():
        n_normal = len(groups['normal'])
        n_anomaly = len(groups['anomaly'])
        total = n_normal + n_anomaly
        
        # 判断模板类型
        if n_normal > 0 and n_anomaly > 0:
            template_type = 'mixed'  # 混合模板
        elif n_anomaly > 0:
            template_type = 'pure_anomaly'  # 纯异常模板
        else:
            template_type = 'pure_normal'  # 纯正常模板
        
        template_analysis[template_id] = {
            'type': template_type,
            'n_normal': n_normal,
            'n_anomaly': n_anomaly,
            'total': total
        }
    
    # 统计各类型模板数量
    type_counts = defaultdict(int)
    for info in template_analysis.values():
        type_counts[info['type']] += 1
    
    print(f"   混合模板（正常+异常）: {type_counts['mixed']}")
    print(f"   纯异常模板: {type_counts['pure_anomaly']}")
    print(f"   纯正常模板: {type_counts['pure_normal']}")
    
    # 第三步：筛选高频模板
    print(f"🔍 筛选高频模板（出现次数 > {min_frequency}）...")
    
    high_freq_templates = {}
    for template_id, info in template_analysis.items():
        if info['total'] > min_frequency:
            high_freq_templates[template_id] = info
    
    print(f"   高频模板数: {len(high_freq_templates)}/{len(template_analysis)}")
    
    # 统计高频模板的类型分布
    high_freq_type_counts = defaultdict(int)
    for info in high_freq_templates.values():
        high_freq_type_counts[info['type']] += 1
    
    print(f"\n📊 高频模板类型分布:")
    print(f"   混合模板: {high_freq_type_counts['mixed']}")
    print(f"   纯异常模板: {high_freq_type_counts['pure_anomaly']}")
    print(f"   纯正常模板: {high_freq_type_counts['pure_normal']}")
    
    # 第四步：根据模板类型选择样本（每个模板最多1-2条）
    print(f"🎯 开始采样（每个模板最多1-2条）...")
    
    ke_samples = []
    
    for template_id in sorted(high_freq_templates.keys()):
        info = high_freq_templates[template_id]
        template_type = info['type']
        groups = template_groups[template_id]
        
        if template_type == 'mixed':
            # 混合模板：选 1条正常 + 1条异常
            normal_sample = random.choice(groups['normal'])
            anomaly_sample = random.choice(groups['anomaly'])
            
            ke_samples.append(normal_sample)
            ke_samples.append(anomaly_sample)
            
        elif template_type == 'pure_anomaly':
            # 纯异常模板：选 1条异常
            anomaly_sample = random.choice(groups['anomaly'])
            ke_samples.append(anomaly_sample)
            
        elif template_type == 'pure_normal':
            # 纯正常模板：选 1条正常
            normal_sample = random.choice(groups['normal'])
            ke_samples.append(normal_sample)
    
    print(f"   采样完成: {len(ke_samples)} 条")
    
    # 第五步：保存为CSV
    print(f"💾 保存 Ke 文件...")
    
    os.makedirs(os.path.dirname(ke_output_file), exist_ok=True)
    
    with open(ke_output_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['template_id', 'label', 'content'])
        
        for sample in ke_samples:
            writer.writerow([
                sample['template_id'],
                sample['label'],
                sample['content']
            ])
    
    print(f"   文件已保存: {ke_output_file}")
    
    # 第六步：统计结果
    print(f"✅ Ke 生成完成")
    
    print(f"📊 最终统计:")
    print(f"   总样本数: {len(ke_samples)}")
    
    # 统计标签分布
    label_counts = defaultdict(int)
    for sample in ke_samples:
        label_counts[sample['label']] += 1
    
    print(f"   正常样本: {label_counts['normal']}")
    print(f"   异常样本: {label_counts['anomaly']}")
    print(f"   覆盖模板数: {len(high_freq_templates)}")
    
    # 详细统计每种类型模板的采样情况
    print(f"📋 各类型模板采样详情:")
    
    type_sample_counts = defaultdict(lambda: {'templates': 0, 'samples': 0, 'normal': 0, 'anomaly': 0})
    
    for template_id in high_freq_templates.keys():
        info = high_freq_templates[template_id]
        template_type = info['type']
        
        # 统计该模板的样本
        template_samples = [s for s in ke_samples if s['template_id'] == template_id]
        
        type_sample_counts[template_type]['templates'] += 1
        type_sample_counts[template_type]['samples'] += len(template_samples)
        
        for sample in template_samples:
            if sample['label'] == 'normal':
                type_sample_counts[template_type]['normal'] += 1
            else:
                type_sample_counts[template_type]['anomaly'] += 1
    
    for template_type in ['mixed', 'pure_anomaly', 'pure_normal']:
        counts = type_sample_counts[template_type]
        if counts['templates'] > 0:
            type_name = {
                'mixed': '混合模板',
                'pure_anomaly': '纯异常模板',
                'pure_normal': '纯正常模板'
            }[template_type]
            
            print(f"\n   {type_name}:")
            print(f"      模板数: {counts['templates']}")
            print(f"      总样本: {counts['samples']} (每个模板{'2条' if template_type == 'mixed' else '1条'})")
            print(f"      正常: {counts['normal']}, 异常: {counts['anomaly']}")
    
    # 计算平均每个模板的样本数
    avg_samples_per_template = len(ke_samples) / len(high_freq_templates)
    print(f"   平均每个模板: {avg_samples_per_template:.2f} 条")
    
    # 显示CSV格式示例
    print(f"📄 CSV格式示例（前20行）:")
    print(f"   template_id,label,content")
    
    for i, sample in enumerate(ke_samples[:20], 1):
        content_preview = sample['content'][:70] + "..." if len(sample['content']) > 70 else sample['content']
        print(f"   {sample['template_id']},{sample['label']},{content_preview}")
    
    # 显示高频模板Top 10
    print(f"\n📋 Top 10 高频模板及其采样:")
    
    sorted_templates = sorted(
        high_freq_templates.items(),
        key=lambda x: x[1]['total'],
        reverse=True
    )[:10]
    
    for i, (template_id, info) in enumerate(sorted_templates, 1):
        # 获取该模板在Ke中的样本
        template_ke_samples = [s for s in ke_samples if s['template_id'] == template_id]
        
        print(f"\n   {i}. [ID:{template_id}] 类型:{info['type']}, 原始总数:{info['total']}")
        print(f"      (原始: 正常:{info['n_normal']}, 异常:{info['n_anomaly']})")
        print(f"      Ke中采样: {len(template_ke_samples)} 条")
        
        for sample in template_ke_samples:
            content_preview = sample['content'][:60] + "..." if len(sample['content']) > 60 else sample['content']
            print(f"         [{sample['label']}] {content_preview}")
    
    return ke_samples


# ==================== 使用示例 ====================

if __name__ == "__main__":
    import random
    import os
    
    # 设置随机种子（可选，用于结果可复现）
    random.seed(42)
    
    # 文件路径
    kh_file = "/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Kh_20k_template_based.csv"
    ke_output_file = "/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Ke_from_kh.csv"
    
    # 生成 Ke
    ke_samples = generate_ke_from_kh(
        kh_file, 
        ke_output_file,
        min_frequency=10  # 只保留出现次数 > 10 的模板
    )
    
    print("\n" + "="*60)
    print("🎉 Ke 生成完成！")
    print("="*60)



🔍 从 Kh 生成 Ke...
   输入文件: /root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Kh_20k_template_based.csv
   最小频率阈值: 10
   采样策略: 每个模板最多 1-2 条样例

📂 读取 Kh 文件...
   读取完成: 1966 个模板
📊 分析模板类型...
   混合模板（正常+异常）: 9
   纯异常模板: 59
   纯正常模板: 1898
🔍 筛选高频模板（出现次数 > 10）...
   高频模板数: 483/1966

📊 高频模板类型分布:
   混合模板: 8
   纯异常模板: 37
   纯正常模板: 438
🎯 开始采样（每个模板最多1-2条）...
   采样完成: 491 条
💾 保存 Ke 文件...
   文件已保存: /root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Ke_from_kh.csv
✅ Ke 生成完成
📊 最终统计:
   总样本数: 491
   正常样本: 446
   异常样本: 45
   覆盖模板数: 483
📋 各类型模板采样详情:

   混合模板:
      模板数: 8
      总样本: 16 (每个模板2条)
      正常: 8, 异常: 8

   纯异常模板:
      模板数: 37
      总样本: 37 (每个模板1条)
      正常: 0, 异常: 37

   纯正常模板:
      模板数: 438
      总样本: 438 (每个模板1条)
      正常: 438, 异常: 0
   平均每个模板: 1.02 条
📄 CSV格式示例（前20行）:
   template_id,label,content
   1,normal,instruction cache parity error corrected
   2,normal,MidplaneSwitchController performing bit sparing on R27-M1-L3-U18-C bit...
   3,normal,generating core.22755
   4,normal,1 ddr

同一条日志内容，在不同上下文中可能是正常或异常
在 BGL 数据集中，同样的日志内容可能在不同时间、不同系统状态下出现：

    正常情况：系统正常运行时偶尔出现的 "data storage interrupt"
    异常情况：系统故障时频繁出现的 "data storage interrupt"

In [3]:
# 验证脚本
def verify_same_content_different_labels(log_file):
    """验证同一内容不同标签的情况"""
    
    content_labels = {}  # {content: {'normal': count, 'anomaly': count}}
    
    with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(None, 9)
            if len(parts) < 10:
                continue
            
            label = parts[0]
            content = parts[9]
            
            is_anomaly = not label.startswith('-')
            label_type = 'anomaly' if is_anomaly else 'normal'
            
            if content not in content_labels:
                content_labels[content] = {'normal': 0, 'anomaly': 0}
            
            content_labels[content][label_type] += 1
    
    # 找出同时有正常和异常标签的内容
    mixed_contents = {
        content: counts 
        for content, counts in content_labels.items() 
        if counts['normal'] > 0 and counts['anomaly'] > 0
    }
    
    print(f"📊 统计结果:")
    print(f"   唯一日志内容总数: {len(content_labels)}")
    print(f"   同时有正常和异常标签的内容: {len(mixed_contents)}")
    print(f"   占比: {len(mixed_contents)/len(content_labels)*100:.2f}%")
    
    # 显示示例
    print(f"📋 示例（同一内容不同标签）:")
    for i, (content, counts) in enumerate(list(mixed_contents.items())[:5], 1):
        print(f"\n   {i}. 内容: {content[:80]}...")
        print(f"      正常出现: {counts['normal']} 次")
        print(f"      异常出现: {counts['anomaly']} 次")

# 运行验证
log_file = "/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/BGL.log"
verify_same_content_different_labels(log_file)


📊 统计结果:
   唯一日志内容总数: 358356
   同时有正常和异常标签的内容: 3
   占比: 0.00%
📋 示例（同一内容不同标签）:

   1. 内容: data storage interrupt...
      正常出现: 2 次
      异常出现: 63491 次

   2. 内容: program interrupt...
      正常出现: 3662 次
      异常出现: 5 次

   3. 内容: Microloader Assertion...
      正常出现: 1 次
      异常出现: 1503 次


# 3.处理Ke_from_ke_new.csv文件，将content列分割为日志列和解释列
## 用于 Embedding 的内容 (Content)：

提取冒号前的部分（或者如果你的数据源里有独立的 Event Template 列，直接用那一列）。
例如：instruction cache parity error corrected
作用：用来做向量化索引，供检索模型（Retrieval Model）使用。
## 用于上下文增强的内容 (Explanation)：

保留整个 content（或者仅保留冒号后的中文解释部分）。
例如：指令缓存奇偶校验错误已通过硬件纠错机制修复, 无系统功能影响...
作用：检索命中后，将这段文本作为 Prompt 的一部分喂给 LLM，帮助 LLM 判断新日志是否异常。

In [13]:
import pandas as pd
import re

def process_csv(file_path):
    try:
        # 1. 读取CSV文件
        # 注意：如果文件没有表头，请添加 header=None, names=['content']
        # 这里假设文件第一行是表头，且包含 'content' 这一列
        df = pd.read_csv(file_path)
        
        # 检查是否存在 content 列
        if 'content' not in df.columns:
            print("错误：CSV文件中未找到 'content' 列。请检查列名。")
            return

        # 2. 使用正则表达式进行分割
        # 逻辑说明：
        # r'(.*)'  : 贪婪匹配，尽可能多地匹配左边的字符（保留原始日志中的时间、代码等）
        # r'[:;]'  : 匹配界限，即冒号或分号
        # r'(.*)'  : 匹配剩余的右边字符（解释说明部分）
        # 使用 extract 可以直接将匹配的两个组生成为两列
        split_data = df['content'].str.extract(r'(.*)[:;](.*)')
        
        # 3. 将分割后的数据赋值给对应的列
        # 左边列（覆盖原content列）
        df['content'] = split_data[0].str.strip() 
        # 右边列（新列 explanation）
        df['explanation'] = split_data[1].str.strip()

        # 4. 打印前几行查看结果
        print("处理完成，前5行数据如下：")
        print(df[['content', 'explanation']].head())

        # 可选：保存到新文件
        df.to_csv('/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/processed_2.csv', index=False, encoding='utf-8-sig')

    except FileNotFoundError:
        print(f"错误：找不到文件 {file_path}")
    except Exception as e:
        print(f"发生错误：{e}")

# 执行函数
if __name__ == "__main__":
    process_csv('/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Ke_from_ke_new.csv')


处理完成，前5行数据如下：
                                             content  \
0           instruction cache parity error corrected   
1  MidplaneSwitchController performing bit sparin...   
2                              generating core.22755   
3  1 ddr errors(s) detected and corrected on rank...   
4  19 L3 EDRAM error(s) (dcr 0x0157) detected and...   

                                         explanation  
0           指令缓存奇偶校验错误已通过硬件纠错机制修复，无系统功能影响，符合正常纠错日志特征  
1  中板交换控制器在R27-M1-L3-U18-C位置的3号比特执行位冗余切换，属于硬件冗余机制...  
2    生成核心转储文件core.22755，为程序调试或异常后分析提供数据支持，属于正常系统诊断输出  
3  在DDR内存0秩、12符号、5比特位置检测到1个错误，已通过ECC纠错机制修复，单次纠错符合...  
4  检测到19个L3 EDRAM错误（DCR寄存器值0x0157），均已通过硬件纠错功能修复，未...  


# 4.构建向量数据库

In [7]:
import pandas as pd
from langchain_community.vectorstores import FAISS
# 确保你安装了 langchain-huggingface 包 (pip install langchain-huggingface)
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_core.documents import Document

# ==========================================
# 修改部分开始：使用 LangChain 的封装类
# ==========================================
model_path = '/root/autodl-tmp/log-anomaly-detection/models/Qwen3-Embedding-0.6B'

# 配置运行参数，建议显式指定用 GPU ('cuda')，否则可能默认用 CPU 会很慢
model_kwargs = {'device': 'cuda'} 
encode_kwargs = {'normalize_embeddings': True} # 归一化通常对余弦相似度检索有帮助

embeddings = HuggingFaceEmbeddings(
    model_name=model_path,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
# ==========================================
# 修改部分结束
# ==========================================

# ================= 构建 Ke 知识库 =================
df_ke = pd.read_csv("/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/processed_2.csv")

ke_docs = []
for index, row in df_ke.iterrows():
    doc = Document(
        # 注意：这里确保 content 是字符串，如果是 NaN 会报错，加个 str() 保险
        page_content=str(row['content']), 
        metadata={
            "explanation": row['explanation'],
            "label": row['label'],
            "source": "Ke"
        }
    )
    ke_docs.append(doc)

# 建立索引
print("正在构建 Ke 向量库 (这可能需要几分钟)...")
vector_store_ke = FAISS.from_documents(ke_docs, embeddings)
vector_store_ke.save_local("/root/autodl-tmp/log-anomaly-detection/output/faiss_ke_index")
print("Ke 向量库构建完成！")

# ================= 构建 Kh 知识库 =================
df_kh = pd.read_csv("/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/Kh_20k_template_based.csv")

# 采样逻辑：每组取前 5 条
df_kh_sampled = df_kh.groupby('template_id').head(5)

kh_docs = []
for index, row in df_kh_sampled.iterrows():
    doc = Document(
        page_content=str(row['content']), # 同样加 str() 保险
        metadata={
            "label": row['label'],
            "original_content": str(row['content']),
            "source": "Kh"
        }
    )
    kh_docs.append(doc)

print("正在构建 Kh 向量库...")
vector_store_kh = FAISS.from_documents(kh_docs, embeddings)
vector_store_kh.save_local("/root/autodl-tmp/log-anomaly-detection/output/faiss_kh_index")
print("Kh 向量库构建完成！")



正在构建 Ke 向量库 (这可能需要几分钟)...
Ke 向量库构建完成！
正在构建 Kh 向量库...
Kh 向量库构建完成！


# 尝试用一下这个数据库，看一看效果

In [10]:
# 1. 定义一个模拟的查询日志 (Query)
# 假设系统新产生了一条日志（纯英文，没有中文解释）
# 这里我们故意稍微改一点点字符，或者用截图里第1行的核心部分，测试它的模糊匹配能力
test_query = "machine check interrupt" 

print(f"🔍 正在检索 Query: [{test_query}]")
print("-" * 50)

# 2. 执行相似度搜索
# k=3 表示返回最相似的前3个结果
# similarity_search_with_score 会同时返回文档对象和距离分数
results = vector_store_ke.similarity_search_with_score(test_query, k=3)

# 3. 打印结果
for i, (doc, score) in enumerate(results):
    print(f"🏆 第 {i+1} 个匹配结果 (距离分数: {score:.4f}):")
    
    # page_content 是我们 Embedding 的部分（应该是英文原日志）
    print(f"   📄 匹配到的历史日志 (Key):       {doc.page_content}")
    
    # metadata 是我们挂载的额外信息（包括中文解释）
    explanation = doc.metadata.get('explanation', '无解释')
    label = doc.metadata.get('label', '未知')
    
    print(f"   💡 检索到的知识 (Value):       {explanation}")
    print(f"   🏷️ 标签: {label}")
    print("-" * 30)



🔍 正在检索 Query: [machine check interrupt]
--------------------------------------------------
🏆 第 1 个匹配结果 (距离分数: 0.0000):
   📄 匹配到的历史日志 (Key):       machine check interrupt
   💡 检索到的知识 (Value):       机器检查中断触发，表明系统检测到硬件级故障（可能涉及CPU、内存或总线组件），需结合硬件状态寄存器进一步定位，属于严重硬件异常，可能导致系统稳定性下降
   🏷️ 标签: anomaly
------------------------------
🏆 第 2 个匹配结果 (距离分数: 0.0000):
   📄 匹配到的历史日志 (Key):       machine check interrupt
   💡 检索到的知识 (Value):       核心异常关键词“machine check interrupt”，对应机器检查中断故障，无正常状态描述，符合BG/L硬件类异常判定规则
   🏷️ 标签: anomaly
------------------------------
🏆 第 3 个匹配结果 (距离分数: 0.3002):
   📄 匹配到的历史日志 (Key):       machine check interrupt (bit=0x06): L3 major internal error
   💡 检索到的知识 (Value):       机器检查中断触发（位标识0x06），具体为L3缓存重大内部错误，表明L3缓存核心功能故障，无法通过硬件纠错修复，属于严重硬件类存储异常，可能导致系统性能大幅下降或程序崩溃
   🏷️ 标签: anomaly
------------------------------


# 5.One-step RAG Approach

In [4]:
import pandas as pd
import json
import time
from tqdm import tqdm
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ================= 配置区域 =================
# 1. 模型路径 (保持和你构建库时一致)
EMBEDDING_MODEL_PATH = '/root/autodl-tmp/log-anomaly-detection/models/Qwen3-Embedding-0.6B'
VECTOR_DB_KE_PATH = "/root/autodl-tmp/log-anomaly-detection/output/faiss_ke_index"
VECTOR_DB_KH_PATH = "/root/autodl-tmp/log-anomaly-detection/output/faiss_kh_index"

# 2. 测试数据集路径 (建议用 Structured CSV 以获取真实标签)
TEST_DATA_PATH = "/root/autodl-tmp/log-anomaly-detection/data/loghub/BGL/BGL_2k.log_structured.csv"

# 3. DeepSeek API 配置
DEEPSEEK_API_KEY = "sk-8ac5bd7db55444d2a47604aaae774443"  # <--- 请替换为你的 DeepSeek API Key
BASE_URL = "https://api.deepseek.com"

# ================= 1. 初始化资源 =================

print("1. 加载 Embedding 模型...")
# 必须使用构建时相同的参数
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True}
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_PATH,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("2. 加载向量数据库 (Ke & Kh)...")
# allow_dangerous_deserialization=True 是必须的，因为我们要加载本地 pickle 文件
vector_store_ke = FAISS.load_local(VECTOR_DB_KE_PATH, embeddings, allow_dangerous_deserialization=True)
vector_store_kh = FAISS.load_local(VECTOR_DB_KH_PATH, embeddings, allow_dangerous_deserialization=True)

print("3. 加载测试数据集...")
# 假设 CSV 中有 'Content' 和 'Label' 列
df_test = pd.read_csv(TEST_DATA_PATH)
# BGL数据集的 Label 列通常是 '-' 表示正常，其他表示异常，或者直接是 'Normal'/'Anomaly'
# 这里做一个简单的预处理，确保 Label 统一为 0 (Normal) 和 1 (Anomaly) 以便计算
# *根据你的实际 CSV 内容调整下面的映射逻辑*
def normalize_label(l):
    l = str(l).lower()
    if 'normal' in l or l == '-':
        return 'Normal'
    return 'Anomaly'

if 'Label' in df_test.columns:
    df_test['GroundTruth'] = df_test['Label'].apply(normalize_label)
else:
    print("警告：测试集中未找到 'Label' 列，无法计算准确率，仅进行推理。")

# ================= 2. 定义核心函数 =================

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=BASE_URL)

def get_enrichment_prompt(log_content):
    """
    检索知识并构建 Prompt
    """
    # 1. 检索 Ke (解释) - 取最相似的 1 条
    results_ke = vector_store_ke.similarity_search(log_content, k=1)
    explanation_text = "No explanation found."
    if results_ke:
        explanation_text = results_ke[0].metadata.get('explanation', '')

    # 2. 检索 Kh (历史) - 取最相似的 3 条
    results_kh = vector_store_kh.max_marginal_relevance_search(log_content, k=5)
    history_text = ""
    for idx, doc in enumerate(results_kh):
        label = doc.metadata.get('label', 'unknown')
        content = doc.page_content
        history_text += f"Example {idx+1}: [Label: {label}] Content: {content}"

    # 3. 构建 Prompt (参考论文格式)
    prompt = f"""You are an expert system log analyst. Your task is to detect if the input log message is Normal or an Anomaly.

                Reference Knowledge (Explanation):
                {explanation_text}

                Reference Knowledge (Historical Similar Logs):
                {history_text}

                Input Log Message:
                "{log_content}"

                Task:
                Based on the reference knowledge and the input log, determine the label.
                Response Format: Only return a JSON object with a single key "label" and value "Normal" or "Anomaly". Do not explain.
                
                Note: Many log messages containing words like 'error' or 'warning' are actually benign system status reports (e.g., corrected errors). Unless there is clear evidence of a system failure or an uncorrected fault, prefer labeling as 'Normal'.
                """
    return prompt

def call_llm(log_content):
    prompt = get_enrichment_prompt(log_content)
    
    try:
        response = client.chat.completions.create(
            model="deepseek-chat", # 或者 deepseek-v3
            messages=[
                {"role": "system", "content": "You are a helpful assistant for log analysis. Always output JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1, # 低温度保证确定性
            response_format={ "type": "json_object" } 
        )
        content = response.choices[0].message.content
        result_json = json.loads(content)
        return result_json.get("label", "Anomaly") # 默认 fallback
    except Exception as e:
        print(f"LLM Error: {e}")
        return "Error"

# ================= 3. 执行推理与评估 =================

# 为了演示和节省API Token，这里只取前 20 条测试，想跑全量把 .head(20) 去掉
#df_subset = df_test.head(20) 
# df_subset = df_test # 跑全量 2k 条

# 随机抽取 1000 条进行测试
# random_state=42 保证每次运行抽到的随机样本是一样的，方便复现结果
if len(df_test) >= 1000:
    df_subset = df_test.sample(n=1000, random_state=42)
else:
    df_subset = df_test # 如果总数少于1000条，就跑全部
    print(f"警告：测试集总数不足1000条，当前将运行全部 {len(df_test)} 条。")

predictions = []
ground_truths = []

print(f"开始推理 {len(df_subset)} 条日志...")

for index, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
    log_content = str(row['Content'])
    
    # 获取真实标签
    gt = row.get('GroundTruth', 'Unknown')
    ground_truths.append(gt)
    
    # 调用 LLM
    pred = call_llm(log_content)
    predictions.append(pred)
    
    # 可选：打印每一条的结果用于调试
    # print(f"Log: {log_content[:50]}... | Pred: {pred} | GT: {gt}")

# ================= 4. 计算指标 =================

# 过滤掉 API 失败的条目
valid_indices = [i for i, p in enumerate(predictions) if p != "Error"]
clean_preds = [predictions[i] for i in valid_indices]
clean_gts = [ground_truths[i] for i in valid_indices]

# 映射为数字计算 (Normal=0, Anomaly=1)
y_pred = [0 if p.lower() == 'normal' else 1 for p in clean_preds]
y_true = [0 if g.lower() == 'normal' else 1 for g in clean_gts]

print("" + "="*30)
print("Evaluation Results")
print("="*30)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}") # 关注异常类的精确率
print(f"Recall:    {recall_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")    # 关注异常类的召回率
print(f"F1 Score:  {f1_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")

print("Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))


1. 加载 Embedding 模型...
2. 加载向量数据库 (Ke & Kh)...
3. 加载测试数据集...
开始推理 1000 条日志...


100%|██████████| 1000/1000 [23:00<00:00,  1.38s/it]

Evaluation Results
Accuracy: 0.9780
Precision: 0.8500
Recall:    0.7969
F1 Score:  0.8226
Detailed Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99       936
     Anomaly       0.85      0.80      0.82        64

    accuracy                           0.98      1000
   macro avg       0.92      0.89      0.91      1000
weighted avg       0.98      0.98      0.98      1000



# 单个日志的加载流程：

1.正在加载 Embedding 模型 (使用 CPU 以防止内存崩溃)...
2. 正在加载向量数据库 (Ke & Kh)...
3. 正在读取测试数据...
为了演示效果，特意抽取了一条【异常(Anomaly)】样本：
============================================================
🚀 开始分析日志
============================================================
📥 [Input Log]: data TLB error interrupt
🏷️ [Ground Truth]: Anomaly
------------------------------
🔍 [Step 1: Retrieving from Ke (Explanation)]
   Found match (Score/Dist: N/A):
   >> 数据转换检测缓冲区（TLB）错误中断，触发硬件异常中断机制，表明虚拟地址到物理地址的转换过程出现故障，可能导致内存访问失效，属于硬件类内存管理异常
------------------------------
📚 [Step 2: Retrieving from Kh (History - MMR Search)]
   [History 1] (Label: anomaly): data TLB error interrupt...
   [History 2] (Label: normal): tlb error.........................0...
   [History 3] (Label: anomaly): instruction TLB error interrupt...
------------------------------
🧠 [Step 3: Constructing Prompt for DeepSeek]
   (Prompt Sent to LLM):You are an expert system log analyst. Your task is to detect if the input log message is Normal or an Anomaly.

                Reference Knowledge (Explanation):
                数据转换检测缓冲区（TLB）错误中断，触发硬件异常中断机制，表明虚拟地址到物理地址的转换过程出现故障，可能导致内存访问失效，属于硬件类内存管理异常

                Reference Knowledge (Historical Similar Logs):
                Example 1: [Label: anomaly] Content: data TLB error interruptExample 2: [Label: normal] Content: tlb error.........................0Example 3: [Label: anomaly] Content: instruction TLB error interrupt

                Input Log Message:
                "data TLB error interrupt"

                Task:
                Based on the reference knowledge and the input log, determine the label.
                Response Format: Only return a JSON object with a single key "label" and value "Normal" or "Anomaly". Do not explain.

                Note: Many log messages containing words like 'error' or 'warning' are actually benign system status reports (e.g., corrected errors). Unless there is clear evidence of a system failure or an uncorrected fault, prefer labeling as 'Normal'.
                
------------------------------
🤖 [Step 4: LLM Inference]
   (Raw LLM Output): {"label": "Anomaly"}
   (Time Taken): 1.71s
============================================================
🏁 Final Result: Anomaly | ✅ Correct
============================================================

# 加载100条数据的情况
1. 加载 Embedding 模型...
2. 加载向量数据库 (Ke & Kh)...
3. 加载测试数据集...
开始推理 100 条日志...
100%|██████████| 100/100 [02:13<00:00,  1.33s/it]
==============================
Evaluation Results
==============================
Accuracy: 0.9600
Precision: 0.6667
Recall:    0.6667
F1 Score:  0.6667
Detailed Classification Report:
              precision    recall  f1-score   support

      Normal       0.98      0.98      0.98        94
     Anomaly       0.67      0.67      0.67         6

    accuracy                           0.96       100
   macro avg       0.82      0.82      0.82       100
weighted avg       0.96      0.96      0.96       100

# 加载1000条数据
1. 加载 Embedding 模型...
2. 加载向量数据库 (Ke & Kh)...
3. 加载测试数据集...
开始推理 1000 条日志...
100%|██████████| 1000/1000 [23:00<00:00,  1.38s/it]
==============================
Evaluation Results
==============================
Accuracy: 0.9780
Precision: 0.8500
Recall:    0.7969
F1 Score:  0.8226
Detailed Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99       936
     Anomaly       0.85      0.80      0.82        64

    accuracy                           0.98      1000
   macro avg       0.92      0.89      0.91      1000
weighted avg       0.98      0.98      0.98      1000